# 3. Spatial Network Generation

## Overview
This notebook constructs **spatial network graphs** representing the EV charging station infrastructure in Germany for each year (2009-2022).

### Network Construction Algorithm
1. **Node creation**: Each charging station becomes a node with lat/lon attributes
2. **Edge creation**: An edge connects two stations if their road distance is ≤ 100 km
   - This 100 km threshold represents the approximate range of early battery electric vehicles (BEVs)
3. **Distance computation**: Uses OSRM API for accurate road distances (with SQLite caching)
4. **Cumulative snapshots**: Each year's network includes all stations commissioned up to that year

### Output
- GraphML files for each year (stored in `results/networks/baseline/`)
- Network visualization PNGs overlaid on Germany's map

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
from shapely.geometry import Point
from pyproj import Transformer
from tqdm import tqdm
import os
import sys

# Add the src directory to the Python path for utility imports
sys.path.insert(0, '../src')
from utils import get_osrm_distance, haversine_distance

## 3.1 Load and Filter Data to German Boundaries

Some entries in the registry have coordinates outside Germany's borders (data entry errors). We use a GeoJSON polygon of Germany to filter these out.

In [ ]:
# Load the cleaned charging station dataset
data = pd.read_csv('../data/processed/ChargingStationCleaned.csv', encoding='utf-8')
data['year'] = pd.to_datetime(data['commissioning_date']).dt.year

# Fix encoding artifacts in federal state names
state_corrections = {
    'Baden-Wï¿½rttemberg': 'Baden-Württemberg',
    'Thï¿½ringen': 'Thüringen',
}
data['federal_state'] = data['federal_state'].replace(state_corrections)

# Create geometry column and convert to GeoDataFrame
data['geometry'] = data.apply(
    lambda row: Point(row['longitude_[dg]'], row['latitude_[dg]']), axis=1
)
gdf = gpd.GeoDataFrame(data, geometry='geometry')

# Load Germany's boundaries and filter stations to within borders
germany_boundary = gpd.read_file('../data/boundaries/4_niedrig.geo.json')
germany_polygon = germany_boundary.geometry.unary_union
gdf = gdf[gdf.geometry.within(germany_polygon)]
data = pd.DataFrame(gdf)

print(f'Stations within Germany: {len(data)}')
data.head()

## 3.2 Network Creation Function

The `create_network()` function builds a proximity-based spatial graph for a given year:
- **Nodes**: All charging stations commissioned up to the target year
- **Edges**: Pairs of stations within `max_distance` km (default: 100 km BEV range)
- **Weights**: Actual road distance in meters (via OSRM API)
- **Projection**: WGS84 → UTM Zone 32N for accurate distance visualization

In [ ]:
def create_network(year, max_distance=100):
    """
    Build a spatial network graph for all charging stations up to a given year.
    
    Args:
        year: Include all stations commissioned on or before this year.
        max_distance: Maximum road distance (km) to create an edge between stations.
    
    Returns:
        None. Saves GraphML and PNG to disk.
    """
    # Filter data to include all stations up to the target year (cumulative)
    data_year = data[data['year'] <= year]
    n = len(data_year)
    positions = data_year[['latitude_[dg]', 'longitude_[dg]']].to_numpy()

    # Transform coordinates from WGS84 to UTM Zone 32N for 2D plotting
    transformer = Transformer.from_crs('epsg:4326', 'epsg:32632', always_xy=True)
    positions_utm = np.array([transformer.transform(x[1], x[0]) for x in positions])

    network = nx.Graph()
    federal_states = data_year['federal_state'].values

    # Add nodes with geographic attributes
    for i in tqdm(range(n), desc=f'Adding nodes ({year})'):
        network.add_node(
            i + 1,
            latitude=positions[i, 0],
            longitude=positions[i, 1],
            federal_state=federal_states[i]
        )

    # Add edges based on road distance threshold
    # Time complexity: O(n²) — necessary for pairwise distance computation
    for i in tqdm(range(n), desc=f'Adding edges ({year})'):
        for j in range(i + 1, n):  # Avoid duplicate edges
            distance = get_osrm_distance(
                positions[i, 0], positions[i, 1],
                positions[j, 0], positions[j, 1]
            )
            if distance is not None and distance < max_distance * 1000:  # Convert km to meters
                network.add_edge(i + 1, j + 1, weight=distance)

    # --- Visualization ---
    pos = {i + 1: positions_utm[i] for i in range(n)}
    options = {'node_color': 'lavender', 'node_size': 10, 'width': 1, 'arrowsize': 1}

    germany_utm = gpd.read_file('../data/boundaries/4_niedrig.geo.json').to_crs('epsg:32632')
    fig, ax = plt.subplots(figsize=(12, 8))
    nx.draw_networkx(network, pos=pos, ax=ax, **options)
    germany_utm.boundary.plot(ax=ax, linewidth=2, color='red', zorder=3)
    plt.title(f'EV Charging Station Network ({year}) — 100 km BEV Range', fontsize=15)
    plt.show()

    # --- Save outputs ---
    os.makedirs('../results/networks/baseline', exist_ok=True)
    os.makedirs('../results/figures/network_graphs', exist_ok=True)
    nx.write_graphml(network, f'../results/networks/baseline/network_{year}_{max_distance}.graphml')
    fig.savefig(f'../results/figures/network_graphs/network_{year}_{max_distance}.png', dpi=300, bbox_inches='tight')
    print(f'Year {year}: {network.number_of_nodes()} nodes, {network.number_of_edges()} edges')

## 3.3 Generate Networks for All Years (2009-2022)

Each call creates a cumulative network snapshot. The computation is O(n²) per year due to pairwise distance calculations, with OSRM results cached to disk.

In [ ]:
# Generate network graphs for each year
# Note: This is computationally expensive. Results are cached in distances.db.
for year in range(2009, 2023):
    create_network(year)